# Generate CDR3 candidates and rank them by summed log-probability`design_cdr3.ipynb` produces exactly one loop: the arg-max residue at each masked position. That throws away everything the model knows about the *second* choice at each position. This notebook keeps the top-k residues per position, enumerates the best whole-loop combinations, and scores each candidate as the sum of its per-residue log-probabilities.It also shows the trap in that score -- and how to get a score that isn't fooled by it.

In [ ]:
import math

import langaai

model = langaai.load()
print(model.device, model.dim)

The same complex as `design_cdr3.ipynb`: a Fab against the SARS-CoV-2 receptor-binding domain (PDB `7wp8`), with its CDR-H3 span taken from IMGT numbering.

In [ ]:
heavy = "QVQLQQPGAELVRPGASVKLSCKASGYTFTSYWMNWVKQRPEQGLEWIGRIDPYDSETHYNQKFKDKAILTVDKSSTTAYMQLSSLTSEDSAVYYCARWGTVEWFFDYWGQGTTLTVSQ"
light = "DIVMTQSPSSLAMSVGQKVTMSCKSSQSLLNSYNQENYLAWYQQKPGQSPKLLVYFASTRESGVPDRFIGSGSGTDFTLTISSVQAEDLADYFCQQHYSTPFTFGSGTKLEIK"
antigen = "CPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGTIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYRYRLFRKSNLKPFERDISTEIYQAGSKPCNGVKGFNCYFPLQSYGFQPTYGVGYQPYRVVVLSFELL"

span = (96, 108)
positions = list(range(*span))
native = heavy[span[0]:span[1]]

ab = model.encode_antibody(heavy, light)
ag = model.embed_antigen(antigen)
masked = ab.mask_region("cdr3", spans=[span])
print(f"native CDR-H3: {native}, {len(positions)} positions masked")

`residue_probabilities` gives the full distribution over amino acids at every masked position, rather than `predict_masked`'s truncated top-k list. `X` ("unresolved residue") is dropped here -- it is a real token in the model's restricted alphabet but not something to put in a designed sequence.

In [ ]:
TOP_K = 4

[probs] = model.residue_probabilities([(masked, ag)])

choices = []  # per position: [(letter, log p), ...] for the TOP_K most probable residues
for pos in positions:
    dist = {a: p for a, p in probs[pos].items() if a != "X"}
    top = sorted(dist.items(), key=lambda kv: -kv[1])[:TOP_K]
    choices.append([(a, math.log(p)) for a, p in top])

print(f"{'pos':>4}  {'native':^6}   top-{TOP_K}")
for pos, letter, opts in zip(positions, native, choices):
    print(f"{pos:>4}  {letter:^6}   " + "  ".join(f"{a}:{math.exp(lp):.2f}" for a, lp in opts))

print(f"\nsearch space: {TOP_K}^{len(positions)} = {TOP_K ** len(positions):,} candidate loops")

Because all positions were masked in **one** forward pass, the model's distribution over the loop factorises: each position's distribution is conditioned on the framework and the antigen, but not on the other masked positions. The score of a whole candidate is therefore just the sum of its per-position log-probabilities.That makes an exact search cheap. Keeping the best `n_candidates` prefixes at each step is enough to find the true top-`n_candidates` sequences over the whole top-k grid -- extending a prefix can only lower its score, so no discarded prefix can come back to win.

In [ ]:
def enumerate_candidates(choices, n_candidates=10):
    """Top-`n_candidates` loops by summed log p, exact over the top-k grid."""
    beam = [("", 0.0)]
    for opts in choices:
        beam = sorted(
            ((seq + a, score + lp) for seq, score in beam for a, lp in opts),
            key=lambda pair: -pair[1],
        )[:n_candidates]
    return beam


candidates = enumerate_candidates(choices, n_candidates=10)


def aar(pred, true):
    return sum(p == t for p, t in zip(pred, true)) / len(true)


native_logp = sum(math.log(probs[pos][a]) for pos, a in zip(positions, native))

print(f"{'rank':>4}  {'candidate':<14}  {'sum log p':>9}  {'per-residue':>11}  {'AAR':>5}")
for i, (seq, score) in enumerate(candidates, 1):
    print(f"{i:>4}  {seq:<14}  {score:>9.3f}  {score / len(positions):>11.3f}  {aar(seq, native):>5.3f}")
print(f"{'--':>4}  {native + ' (native)':<14}  {native_logp:>9.3f}  {native_logp / len(positions):>11.3f}  {1.0:>5.3f}")

### The trapIt is tempting to re-score a candidate with `score_sequence` over the whole span and treat that as an independent check. It isn't one. `score_sequence` masks the positions it scores, so with the *entire* loop masked the forward pass is identical no matter which residues the candidate carries -- the number that comes back is arithmetically the same sum computed above.

In [ ]:
[mean_logp] = model.score_sequence([(ab, ag)], positions=[positions])
print(f"score_sequence over the whole span x {len(positions)} = {mean_logp * len(positions):.4f}")
print(f"summed log p from the single masked pass = {native_logp:.4f}")

### A score that does condition on the rest of the loopTo let a candidate's residues see each other, mask them **one at a time**: build the antibody carrying the candidate loop, then score each position with only that position masked and the other 11 candidate residues visible. Summing those gives the candidate's pseudo-log-likelihood -- the same quantity, but without the independence assumption that produced it.One forward pass per position; `score_sequence` takes a list of pairs, so the whole loop goes in one call.

In [ ]:
def pseudo_log_likelihood(loop):
    """Sum over positions of log p(residue | antigen, rest of the candidate loop)."""
    variant = heavy[:span[0]] + loop + heavy[span[1]:]
    variant_ab = model.encode_antibody(variant, light)
    per_position = model.score_sequence(
        [(variant_ab, ag)] * len(positions),
        positions=[[pos] for pos in positions],
    )
    return sum(per_position)


rescored = sorted(
    ((seq, one_shot, pseudo_log_likelihood(seq)) for seq, one_shot in candidates),
    key=lambda row: -row[2],
)
native_pll = pseudo_log_likelihood(native)

one_shot_rank = {seq: i for i, (seq, _) in enumerate(candidates, 1)}
print(f"{'rank':>4}  {'candidate':<14}  {'sum log p':>9}  {'PLL':>8}  {'was':>4}  {'AAR':>5}")
for i, (seq, one_shot, pll) in enumerate(rescored, 1):
    print(f"{i:>4}  {seq:<14}  {one_shot:>9.3f}  {pll:>8.3f}  {one_shot_rank[seq]:>4}  {aar(seq, native):>5.3f}")
print(f"{'--':>4}  {native + ' (native)':<14}  {native_logp:>9.3f}  {native_pll:>8.3f}  {'--':>4}  {1.0:>5.3f}")

In [ ]:
def spearman(xs, ys):
    def ranks(vals):
        order = sorted(range(len(vals)), key=lambda i: vals[i])
        out = [0.0] * len(vals)
        for rank, i in enumerate(order):
            out[i] = float(rank)
        return out

    rx, ry = ranks(xs), ranks(ys)
    mx, my = sum(rx) / len(rx), sum(ry) / len(ry)
    cov = sum((a - mx) * (b - my) for a, b in zip(rx, ry))
    sx = math.sqrt(sum((a - mx) ** 2 for a in rx))
    sy = math.sqrt(sum((b - my) ** 2 for b in ry))
    return cov / (sx * sy)


rho = spearman([row[1] for row in rescored], [row[2] for row in rescored])
print(f"Spearman rho between the two rankings: {rho:.3f}  (n={len(rescored)} candidates)")

### What to take from this, and what not to- **The two scores rank differently.** The summed log-probability is what selects the candidate set, but it cannot see interactions between positions; the pseudo-log-likelihood can, and it reshuffles the order. Generate with the first, rank with the second.- **Top-k search collapses into a motif.** Look at the candidate list: the same few residues recur, because the highest-probability options at neighbouring positions are correlated and the search has no diversity term. If you need a diverse panel rather than the ten nearest variants of one loop, sample from the per-position distributions instead of enumerating their modes, or enforce a minimum pairwise distance while you search.- **Neither score is affinity.** Both are masked-reconstruction likelihoods. Nothing in this project's evaluation work connected them to binding. That the native loop can score *below* the model's own designs on its own likelihood scale is the clearest possible statement of the gap -- the native loop is the one that provably binds. The one affinity path validated end to end is a regressor fitted on `cls_embedding`.